In [ ]:
#Install required dependencies
%pip install -r requirements.txt

In [ ]:
#dotenv module
from dotenv import load_dotenv
_ = load_dotenv()

In [ ]:
#Adding typing module for typed python
# Documentation for python typing annotation https://docs.python.org/3/library/typing.html
from typing import TypedDict, Annotated, List
import operator

In [ ]:
#langgraph module and tavily search tool
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AnyMessage, SystemMessage, ToolMessage, HumanMessage
from langchain_tavily import TavilySearch

In [ ]:
# Google Gemini used for this example : https://python.langchain.com/docs/integrations/providers/google/
from langchain_google_genai import ChatGoogleGenerativeAI

tavily = TavilySearch(max_results=2)
model = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int

In [ ]:
PLAN_PROMPT = """You are an expert writer tasked with writing a high level outline of an essay. 
Write such an outline for the user provided topic. Give an outline of the essay along with any relevant notes 
or instructions for the sections."""


In [ ]:
WRITER_PROMPT = """You are an essay assistant tasked with writing excellent 5-paragraph essays.
Generate the best essay possible for the user's request and the initial outline. 
If the user provides critique, respond with a revised version of your previous attempts. 
Utilize all the information below as needed: 

------

{content}"""

In [ ]:
REFLECTION_PROMPT = """You are a teacher grading an essay submission. 
Generate critique and recommendations for the user's submission. 
Provide detailed recommendations, including requests for length, depth, style, etc."""

In [ ]:
RESEARCH_PLAN_PROMPT = """You are a researcher charged with providing information that can 
be used when writing the following essay. Generate a list of search queries that will gather 
any relevant information. Only generate 3 queries max."""

In [ ]:
RESEARCH_CRITIQUE_PROMPT = """You are a researcher charged with providing information that can 
be used when making any requested revisions (as outlined below). 
Generate a list of search queries that will gather any relevant information. Only generate 3 queries max."""


(A Pydantic model is an object, similar to a Python dataclass, that defines and stores data about an entity with annotated fields. Unlike dataclasses, Pydantic's focus is centered around automatic data parsing, validation, and serialization.)

In [ ]:
from langchain_core.pydantic_v1 import BaseModel

class Queries(BaseModel):
    queries: List[str]

First, we generate the first node that is going to write the plan asked by the user

In [ ]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

Then, we generate the research plan node that is going to receive the outline for an essay and generate the research we will need to to troughout the search tool.

PD: we are forcing the model to use Pydantic models 

In [ ]:
def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke(input=[
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.invoke({'query': q})
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

In [ ]:
def generation_node(state: AgentState):
    print(f"Calling generation... iteration{state.get("revision_number")}")
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    messages = [
        SystemMessage(content=WRITER_PROMPT.format(content=content)),
        user_message
    ]
    response = model.invoke(messages)
    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 1) +1
    }

In [ ]:
def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

In [ ]:
def research_critique_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

In [ ]:
def should_continue(state):
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

In [ ]:
builder.set_entry_point("planner")
builder.add_conditional_edges(
    "generate",
    should_continue,
    {END: END, "reflect": "reflect"}
)
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")
builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

graph = builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [24]:
thread = {"configurable": {"thread_id": "1"}}
for s in graph.stream({
    "task": "what is the difference between langchain and langsmith",
    "max_revisions": 2,
    "revision_number": 1,
    "content": []
}, thread):
    print(s)

{'planner': {'plan': 'Of course. Here is a high-level essay outline on the difference between LangChain and LangSmith, complete with notes and instructions for each section.\n\n***\n\n### **Essay Topic: A Tale of Two Tools: Distinguishing LangChain the Framework from LangSmith the Platform**\n\n**Essay Goal:** To clearly articulate that LangChain and LangSmith are not competing products, but rather complementary tools that serve distinct purposes at different stages of the Large Language Model (LLM) application lifecycle. LangChain is for *building*, and LangSmith is for *observing, debugging, and evaluating*.\n\n---\n\n### **High-Level Essay Outline**\n\n#### **I. Introduction: The Rise of the LLM Application Stack**\n\n*   **A. Hook:** Start with the explosion of interest in building applications powered by LLMs. Highlight the initial excitement followed by the realization that building robust, reliable, and production-ready LLM apps is a significant engineering challenge.\n*   **B. 

TypeError: BaseTool.invoke() missing 1 required positional argument: 'input'

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from helper import ewriter, writer_gui

In [ ]:
MultiAgent = ewriter()
app = writer_gui(MultiAgent.graph)
app.launch()